### Import Libraries

In [15]:
import numpy as np
import pandas as pd
import re
import string

### Load News Datasets

In [16]:
true = pd.read_csv('True.csv')
fake = pd.read_csv('Fake.csv')

### Load LIAR Dataset

In [17]:
liar_train = pd.read_csv('train.csv')
liar_test  = pd.read_csv('test.csv')
liar_valid = pd.read_csv('valid.csv')

liar_df = pd.concat([liar_train, liar_test, liar_valid])
liar_df = liar_df[['statement', 'label']]
liar_df.rename(columns={'statement': 'text'}, inplace=True)

def convert_label(label):
    if label in ['true', 'mostly-true']:
        return 1
    else:
        return 0

liar_df['label'] = liar_df['label'].apply(convert_label)

### Merge Fake and True into news_df

In [18]:
true['label'] = 1
fake['label'] = 0

news_df = pd.concat([fake, true], axis=0)
news_df = news_df.drop(['title', 'subject', 'date'], axis=1)

### Combine news_df and liar_df

In [19]:
combined_df = pd.concat([news_df, liar_df])

combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print('Label distribution after shuffle:')
print(combined_df['label'].value_counts())

Label distribution after shuffle:
label
0    46443
1    21417
Name: count, dtype: int64


### Balance Classes

In [20]:
from sklearn.utils import resample

df_fake = combined_df[combined_df.label == 0]
df_true = combined_df[combined_df.label == 1]

df_fake_downsampled = resample(df_fake,
                               replace=False,
                               n_samples=len(df_true),
                               random_state=42)

combined_df = pd.concat([df_fake_downsampled, df_true])


combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(combined_df['label'].value_counts())

label
0    21417
1    21417
Name: count, dtype: int64


### Text Preprocessing Function

In [21]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def wordopt(text):
    text = text.lower()
    # remove urls
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # remove html
    text = re.sub(r'<.*?>', '', text)
    # remove punctuation
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
  
    text = re.sub(r'\w*\d\w*', '', text)
    # tokenize, remove stopwords, lemmatize
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

combined_df['text'] = combined_df['text'].apply(wordopt)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Train-Test Split

In [22]:
from sklearn.model_selection import train_test_split

x = combined_df['text']
y = combined_df['label']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

print('Train size:', x_train.shape)
print('Test size:', x_test.shape)

Train size: (29983,)
Test size: (12851,)


### TF-IDF Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_instance = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

xv_train = vectorizer_instance.fit_transform(x_train)
xv_test  = vectorizer_instance.transform(x_test)

print('xv_train shape:', xv_train.shape)
print('xv_test shape :', xv_test.shape)

xv_train shape: (29983, 10000)
xv_test shape : (12851, 10000)


### Logistic Regression

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

LR = LogisticRegression(max_iter=1000)
LR.fit(xv_train, y_train)

pred_lr = LR.predict(xv_test)

print('Accuracy:', accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))

Accuracy: 0.9897284258034394
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      6453
           1       0.99      0.99      0.99      6398

    accuracy                           0.99     12851
   macro avg       0.99      0.99      0.99     12851
weighted avg       0.99      0.99      0.99     12851



### Helper Functions

In [25]:
def output_label(n):
    if n == 1:
        return " It is Genuine News"
    elif n == 0:
        return " It is Fake News"

def clean_input(news):
    lines = news.split("\n")
    valid_lines = [line for line in lines if len(line.split()) > 5]
    return " ".join(valid_lines)

def manual_testing(news):
    # Step 1: remove short/ad lines
    news = clean_input(news)

    # Step 2: preprocess text
    cleaned_news = wordopt(news)

    # Step 3: guard against empty input
    if len(cleaned_news.strip()) == 0:
        return " Invalid or insufficient news text"

    # Step 4: vectorize
    news_vec = vectorizer_instance.transform([cleaned_news])

    # Step 5: guard against zero features (out-of-vocabulary input)
    if news_vec.nnz == 0:
        return " Input not understood by model (no known vocabulary found)"

    # Step 6: predict
    pred = LR.predict(news_vec)
    return output_label(pred[0])

### Test the Model

In [26]:
news_article = str(input("Paste your news article here: "))
print(manual_testing(news_article))

Paste your news article here:  Former Australian special forces soldier Ben Roberts-Smith has been arrested at Sydney airport and is expected to face charges for alleged war crimes committed in Afghanistan, according to the Australian Broadcasting Corporation (ABC).  The 47-year-old was expected to appear in a court in New South Wales later on Tuesday over five counts of the war crime of murder, related to unarmed Afghan nationals who “were not taking part in hostilities at the time of their alleged murder”, Australian Federal Police Commissioner Krissy Barrett told reporters in Sydney on Tuesday, according to the ABC.  Recommended Stories list of 4 items list 1 of 4Australia bans visitors from Iran amid war in the Middle East list 2 of 4Australia’s post-Bondi crackdown accused of targeting pro-Palestinian voices list 3 of 4Albanese says Australia playing “constructive” role in the war on Iran list 4 of 4Court rejects Australian soldier’s defamation appeal over Afghan killings end of l

 It is Genuine News


### Save Model and Vectorizer

In [27]:
import joblib

joblib.dump(LR, 'logistic_regression_model.joblib')
print('Logistic Regression model saved.')

joblib.dump(vectorizer_instance, 'tfidf_vectorizer.joblib')
print('TF-IDF Vectorizer saved.')

Logistic Regression model saved.
TF-IDF Vectorizer saved.
